In [1]:
import pandas as pd
import sys
sys.path.append("..")
from config import REGION_PROCESSED_DIR

# 1. read the CLEAN files
streets = pd.read_csv(REGION_PROCESSED_DIR / "streets_clean.csv")
pavement = pd.read_csv(REGION_PROCESSED_DIR / "pavement_clean.csv")

# 2. each segment has 2 ends
node1 = streets[["cnn", "classcode", "f_node_cnn"]].rename(columns={"f_node_cnn": "node"})
node2 = streets[["cnn", "classcode", "t_node_cnn"]].rename(columns={"t_node_cnn": "node"})
seg_node = pd.concat([node1, node2], ignore_index=True) #2 rows per segment

# 3.  add segment PCI
seg_node = seg_node.merge(pavement[["cnn", "pci_score"]], on="cnn", how="left")

# 4.  group by intersection
nodes_df = seg_node.groupby("node").agg(
    n_segments=("cnn", "nunique"), #  number of segments
    main_class=("classcode", "min"), #  lowest class (assumption)
    pci_seg_mean=("pci_score", "mean"), #  mean PCI
    pci_seg_min=("pci_score", "min"), #  min PCI
).reset_index()

# 5.  direct PCI of the intersection
pci_node = pavement[["cnn", "pci_score"]].rename(columns={"cnn": "node", "pci_score": "pci_node"})
nodes_df = nodes_df.merge(pci_node, on="node", how="left")

# 6.  checks
print(len(nodes_df))
print("pci_node:", nodes_df["pci_node"].notna().mean())
print("pci_seg_mean:", nodes_df["pci_seg_mean"].notna().mean())
print("no PCI at all:", (nodes_df["pci_node"].isna() & nodes_df["pci_seg_mean"].isna()).mean())
print(nodes_df["n_segments"].value_counts().sort_index())

# 7.  save
nodes_df.to_csv(REGION_PROCESSED_DIR / "intersections.csv", index=False)





9822
pci_node: 0.49572388515577276
pci_seg_mean: 0.8649969456322542
no PCI at all: 0.13490124210954998
n_segments
1     1110
2      490
3     3760
4     3562
5      391
6      435
7       47
8       18
9        4
10       5
Name: count, dtype: int64


In [2]:
crashes_clean = pd.read_csv(REGION_PROCESSED_DIR / "crashes_clean.csv")
small = nodes_df[nodes_df["n_segments"] <= 2]["node"]
print(crashes_clean["intersection_id"].isin(small).mean())
print(crashes_clean["intersection_id"].isin(nodes_df["node"]).mean())

0.003124544337284146
0.999791697044181


In [3]:
nodes_df = nodes_df[nodes_df["n_segments"] >= 3].copy() #  real intersections only
print(len(nodes_df))

8222


In [4]:
nodes_df["pci"] = nodes_df["pci_node"].fillna(nodes_df["pci_seg_mean"])


In [6]:
nodes_df = nodes_df[nodes_df["n_segments"] >= 3].copy()
nodes_df["pci"] = nodes_df["pci_node"].fillna(nodes_df["pci_seg_mean"])
nodes_df.to_csv(REGION_PROCESSED_DIR / "intersections.csv", index=False)

print(nodes_df["pci"].notna().mean())

0.9151058136706397


In [8]:
nodes_df.to_csv(REGION_PROCESSED_DIR / "intersections.csv", index=False)